This notebook trains recommendation models on real user-item interaction data and reports their performance.

It reads the optimal hyperparameters for each dataset from `opt_params.json` (generated in `01_tuning.ipynb`) and generates:
- `results/runs/<dataset>/clean/metrics_top<N>.csv`: aggregated metric scores
- `results/runs/<dataset>/clean/ndcg.npz`: per-user NDCG scores for hypothesis testing
- `results/runs/<dataset>/clean/ndcg_index.csv`: metadata for the stored per-user NDCG arrays

The notebook resumes safely: set `MODEL_IDS_TO_RUN` to a subset such as `["neumf"]` when adding one model without replacing prior results.

Ratios are reported with the unprivileged group (numerator) defined as the group with the lowest NDCG score.

**Model naming**: Fairness-aware models are trained using group indicators based on a sensitive attribute, and their names include that attribute as a suffix. 
For example, `mf-absolute-age` denotes that MF-absolute was trained on age groups, whereas `mf-absolute-gender` denotes it was trained on gender groups.

In [1]:

import json
from pathlib import Path

import pandas as pd
import numpy as np

from src.helper_functions import paths
from src.helper_functions.data_loader import *
from src.helper_functions.data_splitter import *
from src.helper_functions.train_utils import *
from src.helper_functions.metrics_accuracy import *
from src.helper_functions.result_utils import (
    add_group_assignments_from_row,
    add_group_ratios,
    load_checkpoint,
    log_progress,
    make_run_key,
    store_user_labels,
    training_heartbeat,
    write_checkpoint,
)

In [ ]:
# Configuration
DATASETS = ["ml-100k", "ml-1m", "lastfm-1k", "ftky", "fnyc"]
SEED = 42
RATING_THRES = 4
LIST_SIZE = 10

METRICS = {"precision": precision_at_n, "recall": recall_at_n, "ndcg": tndcg_at_n}
GROUPS_CONFIG = {"gender": ["f", "m"], "age": ["y", "o"]}

# Use None for all models, or a subset such as ["neumf"] / ["mf-over-gender"].
MODEL_IDS_TO_RUN = ["neumf"]

headers = [
    "run_key",
    "dataset",
    "model",
    *METRICS.keys(),
    *[f"{m}_{g}" for groups in GROUPS_CONFIG.values() for g in groups for m in METRICS],
    *[f"{m}_{attr}_ratio" for attr in GROUPS_CONFIG for m in METRICS],
]

CLEAN_RESUME_KEY_COLUMNS = ["dataset", "model"]
PROGRESS_LOG_PATH = paths.log_file("02_train_real.log")
HEARTBEAT_SECONDS = 60


In [ ]:
selected_model_ids = None if MODEL_IDS_TO_RUN is None else set(MODEL_IDS_TO_RUN)


def model_is_selected(model_name, model_tag=None):
    if selected_model_ids is None:
        return True
    return model_name in selected_model_ids or model_tag in selected_model_ids


for dataset in DATASETS:
    data, groups_gender, groups_age = load_dataset_by_name(dataset)
    folder = paths.dataset_dir(dataset)

    with open(paths.opt_params(dataset), "r") as f:
        opt_params = json.load(f)

    R_train, R_val, R_test, uid_to_index, iid_to_index = chronological_split_per_user(data)
    R_train_full = R_train + R_val
    log_progress(
        f"Prepared {dataset}: "
        f"train_shape={R_train.shape}, train_nnz={R_train.nnz}, "
        f"val_nnz={R_val.nnz}, test_nnz={R_test.nnz}, "
        f"train_full_nnz={R_train_full.nnz}",
        PROGRESS_LOG_PATH,
    )

    groups_map = {
        "gender": map_user_indices(groups_gender, uid_to_index),
        "age": map_user_indices(groups_age, uid_to_index),
    }

    store_user_labels(str(folder), R_test, groups_map)

    pre_attack_ndcg_dir = paths.pre_attack_dir(dataset)
    pre_attack_ndcg_dir.mkdir(parents=True, exist_ok=True)
    result_path = paths.clean_results(dataset, LIST_SIZE)
    ndcg_index_path = pre_attack_ndcg_dir / "ndcg_index.csv"
    ndcg_path = pre_attack_ndcg_dir / "ndcg.npz"

    results, ndcg_index, ndcg_store, run_id, completed = load_checkpoint(
        result_path, ndcg_index_path, ndcg_path, CLEAN_RESUME_KEY_COLUMNS
    )

    base_models = initialize_base_models(opt_params, SEED)
    fair_models = initialize_fair_models(opt_params, SEED)

    group_assignments = {}
    for row in results:
        add_group_assignments_from_row(group_assignments, row["model"], row, GROUPS_CONFIG)

    for model_name, model in base_models.items():
        if not model_is_selected(model_name):
            continue

        current_key = make_run_key(dataset, model_name)
        if current_key in completed:
            log_progress(f"Skipping completed clean {dataset}/{model_name}", PROGRESS_LOG_PATH)
            continue

        train_label = f"clean {dataset}/{model_name}"
        log_progress(
            f"Training {train_label}: "
            f"neumf_validation={model_name == 'neumf'}, "
            f"train_nnz={R_train.nnz if model_name == 'neumf' else R_train_full.nnz}, "
            f"val_nnz={R_val.nnz if model_name == 'neumf' else 0}, "
            f"mask_nnz={R_train_full.nnz}",
            PROGRESS_LOG_PATH,
        )
        with training_heartbeat(train_label, PROGRESS_LOG_PATH, HEARTBEAT_SECONDS):
            if model_name == "neumf":
                R_hat = train_model(
                    model, R_train, R_test,
                    val_matrix=R_val,
                    mask_matrix=R_train_full,
                )
            else:
                R_hat = train_model(model, R_train_full, R_test)
        log_progress(f"Finished fit {train_label}; computing metrics", PROGRESS_LOG_PATH)

        ndcg_scores = tndcg_at_n(R_hat, R_test, RATING_THRES, LIST_SIZE)
        key = current_key
        ndcg_store[key] = ndcg_scores
        ndcg_index.append({"key": key, "dataset": dataset, "model": model_name})

        row = {"run_key": current_key, "dataset": dataset, "model": model_name}
        for attr, group_indices in groups_map.items():
            if not group_indices:
                continue

            metrics = compute_metrics(
                R_hat, R_test, RATING_THRES, LIST_SIZE, METRICS, groups=group_indices
            )
            row.update(metrics)

            g1, g2 = GROUPS_CONFIG[attr]
            ndcg_g1 = metrics.get(f"ndcg_{g1}")
            ndcg_g2 = metrics.get(f"ndcg_{g2}")
            unpriv, priv = (g1, g2) if ndcg_g1 < ndcg_g2 else (g2, g1)
            group_assignments.setdefault(model_name, {})[attr] = (unpriv, priv)
            row = add_group_ratios(row, attr, unpriv, priv, METRICS, GROUPS_CONFIG)

        results.append(row)
        completed.add(current_key)
        write_checkpoint(result_path, ndcg_index_path, ndcg_path, results, ndcg_index, ndcg_store, headers)
        log_progress(f"Checkpointed clean {dataset}/{model_name}", PROGRESS_LOG_PATH)

    for attr, group_indices in groups_map.items():
        if not group_indices:
            continue

        for model_name, model in fair_models.items():
            model_tag = model_result_name("fair", model_name, attr)
            if not model_is_selected(model_name, model_tag):
                continue

            current_key = make_run_key(dataset, model_tag)
            if current_key in completed:
                log_progress(f"Skipping completed clean {dataset}/{model_tag}", PROGRESS_LOG_PATH)
                continue

            base_model = get_base_model_for(model_name)
            if attr not in group_assignments.get(base_model, {}):
                raise RuntimeError(
                    f"Missing clean group assignment for {dataset}/{base_model}/{attr}. "
                    "Run the corresponding base model first."
                )

            unpriv, priv = group_assignments[base_model][attr]
            train_label = f"clean {dataset}/{model_tag}"
            log_progress(
                f"Training {train_label}: train_full_nnz={R_train_full.nnz}, "
                f"unprivileged_users={len(groups_map[attr][unpriv])}",
                PROGRESS_LOG_PATH,
            )
            with training_heartbeat(train_label, PROGRESS_LOG_PATH, HEARTBEAT_SECONDS):
                R_hat = train_model(
                    model, R_train_full, R_test,
                    unprivileged_users=groups_map[attr][unpriv]
                )
            log_progress(f"Finished fit {train_label}; computing metrics", PROGRESS_LOG_PATH)

            ndcg_scores = tndcg_at_n(R_hat, R_test, RATING_THRES, LIST_SIZE)
            key = current_key
            ndcg_store[key] = ndcg_scores
            ndcg_index.append({"key": key, "dataset": dataset, "model": model_tag})

            metrics = compute_metrics(
                R_hat, R_test, RATING_THRES, LIST_SIZE, METRICS, groups=group_indices
            )
            metrics = add_group_ratios(metrics, attr, unpriv, priv, METRICS, GROUPS_CONFIG)
            row = {"run_key": current_key, "dataset": dataset, "model": model_tag}
            row.update(metrics)

            results.append(row)
            completed.add(current_key)
            write_checkpoint(result_path, ndcg_index_path, ndcg_path, results, ndcg_index, ndcg_store, headers)
            log_progress(f"Checkpointed clean {dataset}/{model_tag}", PROGRESS_LOG_PATH)
